In [16]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
from langchain.agents import create_agent
from langchain.messages import HumanMessage, ToolMessage, AIMessage
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from src.agentic.tools import NOTEBOT_TOOLS
from dotenv import load_dotenv
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command
from langchain.agents.middleware import ToolRetryMiddleware
from langchain.agents.middleware import (wrap_tool_call,before_model,after_model,AgentState)
from collections.abc import Callable
from langchain.tools.tool_node import ToolCallRequest
from langgraph.runtime import Runtime
from typing import Any
from langgraph.stream import StreamTransformer, StreamChannel
import pprint
pprint = pprint.pprint

In [17]:
load_dotenv()

True

In [18]:
model = ChatOllama(
    model="gemma4:e2b-mlx",
    reasoning=False,
    temperature=0.0,
    top_p=0.5
)

In [19]:
model = ChatOllama(
    model="gpt-oss:20b-cloud",
    base_url=os.getenv("OLLAMA_BASE_URL"),
    headers={"Authorization": f"Bearer {os.getenv('OLLAMA_API_KEY')}"},
    temperature=0
)

In [5]:
@wrap_tool_call
def log_tool(request: ToolCallRequest, handler: Callable[[ToolCallRequest], ToolMessage | Command],) -> ToolMessage | Command:
    print(f"Executing tool: {request.tool_call['name']}")
    print(f"Arguments: {request.tool_call['args']}")
    try:
        result = handler(request)
        print(f"Tool result:{result.content[:40]}...")
        return result
    except Exception as e:
        print(f"Tool failed: {e}")
        raise

In [6]:
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langchain_core.messages import RemoveMessage
@before_model
def log_and_trim_before_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    messages=state['messages']
    if len(messages)>30:
        first_msg = messages[0]
        recent_messages = messages[-29:]
        new_messages = [first_msg] + recent_messages
        print(f"About to call model with {len(new_messages)} messages")
        return {
            "messages" : [
                RemoveMessage(id=REMOVE_ALL_MESSAGES),
                *new_messages
            ]
        }
    print(f"About to call model with {len(state['messages'])} messages")
    return None

In [7]:
@after_model
def log_after_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    last = state["messages"][-1]

    if not isinstance(last, AIMessage):
        return None

    for tool_call in last.tool_calls:
        if tool_call["name"] == "search_notes":
            print("Search tool intercepted.")
            return Command(
                update={
                    "messages": [
                        ToolMessage(
                            content="Search is currently unavailable. Try again after 5mins",
                            tool_call_id=tool_call["id"],
                        )
                    ]
                }
            )

    # No interception → continue normally
    return None

In [8]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

checkpointer = SqliteSaver(sqlite3.connect("../db/checkpoint.db",check_same_thread=False))

In [9]:
conn = sqlite3.connect("../db/checkpoint.db",check_same_thread=False)
conn.execute(
    "DELETE FROM checkpoints WHERE thread_id = 't5'"
)
conn.commit()

In [10]:
import sqlite3
from langgraph.store.sqlite import SqliteStore

conn = sqlite3.connect("../db/store.db",autocommit=True,check_same_thread=False)
store = SqliteStore(conn=conn)

In [102]:
fruits = [
("apple","""Few cultivated plants have traveled a journey as remarkable as the apple. The story of this fruit begins in the rugged Tian Shan Mountains of Central Asia, where forests of wild apples still grow today. The origin of nearly every modern apple can be traced back to these forests, where bears, horses, and other animals spread seeds long before humans cultivated orchards. As merchants carried apples along the Silk Road, trees naturally crossed with local relatives, creating extraordinary genetic diversity. Human preference then became a powerful evolutionary force. Farmers repeatedly chose trees that produced larger, sweeter, and crisper fruit, gradually transforming small tart apples into the familiar dessert varieties found today. Their brilliant red skin became increasingly common because anthocyanin pigments appealed to both wildlife and people, while green and yellow cultivars preserved older pigment combinations. Modern apples are therefore a partnership between natural evolution and thousands of years of unconscious selective breeding. An interesting fact is that apples belong to the rose family, making them unexpectedly close relatives of pears, peaches, cherries, and even ornamental roses."""),
("pear","""Walk through an old European woodland and you might encounter the ancestors of the modern pear. Unlike apples, this fruit evolved a flesh filled with tiny stone cells that discourage animals from eating it before ripening. The origin of cultivated pears stretches across Europe and western Asia, where many different wild species slowly adapted to diverse climates. Ancient growers noticed that some trees naturally produced softer, sweeter fruit and propagated only those individuals. Over centuries the taste shifted from sharply astringent to delicately sweet without losing its characteristic texture. Although green remains the classic appearance, evolution and breeding introduced bronze, yellow, and crimson skins through changing pigment combinations. Pear trees themselves are remarkable survivors, with some historic orchards containing specimens that continue producing fruit after more than a century. Their wood is equally famous among craftsmen because it remains exceptionally stable after drying."""),
("quince","""Long before apples dominated European orchards, another fruit enjoyed celebrity status. Quince, whose origin lies around the Caucasus and western Asia, filled ancient homes with fragrance so intense that ripe specimens were often placed indoors simply to perfume a room. Rather than evolving into a fruit that begged to be eaten fresh, quince followed a different evolutionary strategy. Dense flesh packed with pectin and aromatic compounds protected the seeds while creating a culinary treasure once heat unlocked its sweetness. During cooking, pale flesh transforms into a beautiful rosy color through complex chemical reactions rather than simple pigments. Greek and Roman writers praised quince for both its beauty and its perfume, and some historians suspect that several references to mythical golden apples actually described this remarkable fruit. Even today, its powerful aroma remains one of its defining evolutionary signatures."""),
("loquat","""Most fruit trees spend winter dormant, patiently waiting for spring blossoms. Loquat ignores that rule entirely. This unusual fruit, whose origin is southeastern China, flowers in autumn and quietly develops throughout winter before ripening in spring. That strange schedule allowed it to occupy an ecological niche with little competition from neighboring trees. Although its clusters resemble small citrus fruits, loquat belongs firmly within the same evolutionary family as apples and pears. Bright orange flesh results from carotenoid pigments that become increasingly concentrated as the fruit ripens. Wild loquats were noticeably more bitter than cultivated forms, but centuries of selection emphasized sweetness while preserving refreshing acidity. The tree's willingness to bloom during cool weather has fascinated botanists for generations and remains one of its most distinctive evolutionary adaptations."""),
("peach","""If paleontologists had discovered only fossil peach stones, they could still have guessed that the surrounding fruit once relied on large animals. The massive pit protecting the seed evolved millions of years before agriculture, ensuring survival even after passing through powerful jaws. The origin of cultivated peaches lies in China, yet fossil evidence suggests peach-like ancestors existed there long before humans appeared. Early fruit contained little flesh surrounding the stone, but careful cultivation steadily enlarged the edible portion while increasing sweetness and aroma. Today the characteristic blush of ripe peaches comes from anthocyanins layered over golden carotenoids, producing colors that signal perfect ripeness. Curiously, the peach shares a surprisingly close relationship with almonds, explaining why crushed peach kernels carry a faint almond fragrance."""),
("plum","""Evolution rarely follows neat boundaries, and few fruit illustrate that better than the plum. Instead of descending from a single straightforward lineage, many plum species naturally crossed with one another wherever their ranges overlapped. The origin of cultivated plums therefore spans Europe and Asia, creating extraordinary diversity. Some evolved into tiny tart fruit suited for wildlife, while others gradually became large dessert varieties through human selection. Purple skins owe their richness to anthocyanins, yellow cultivars emphasize carotenoids, and green varieties retain chlorophyll even when ripe. Taste evolved just as dramatically, ranging from mouth-puckering acidity to syrup-like sweetness. Because different plum species hybridize so readily, breeders continue creating new varieties that combine traits from several ancient lineages."""),
("apricot","""The apricot tells a story of endurance more than abundance. This fruit originated across Central Asia and northwestern China, regions where winters could be severe and summers intensely dry. Trees that survived those extremes passed their resilience to future generations. Ancient traders later carried apricots westward along caravan routes because dried fruit remained nutritious for months, making them ideal travel provisions. Evolution favored brilliant orange flesh rich in carotenoids, pigments that also produce vitamin A precursors. While wild apricots often leaned toward tartness, cultivation gradually balanced acidity with honey-like sweetness without sacrificing their distinctive floral aroma. Even today, dried apricots preserve much of the character that made them valuable companions on ancient journeys."""),
("cherry","""Every spring, clouds of white blossoms announce the beginning of another cherry season, yet the true evolutionary success of this fruit comes months later. Birds are irresistibly drawn to the bright red and nearly black colors of ripe cherries, carrying seeds across landscapes far beyond the parent tree. The origin of cultivated cherries stretches across Europe and western Asia, where both sweet and sour species evolved under different ecological pressures. Human growers eventually separated these lineages, enhancing sweetness in dessert cherries while preserving acidity in culinary varieties. Their delicate flavor depends on a surprisingly large collection of aromatic molecules rather than sugar alone. Long after the fruit disappears each year, cherry blossoms continue serving as enduring symbols of renewal in many cultures."""),
("orange","""The family history of the orange resembles a genealogical puzzle more than a straight line. This fruit is not the direct descendant of a single wild ancestor. Instead, its origin lies in ancient hybridization between mandarin and pomelo somewhere in southern China or northeastern India. Evolution combined the sweetness of one parent with the refreshing acidity of the other, producing a remarkably balanced fruit. As ripening proceeds, green chlorophyll disappears and carotenoid pigments create the familiar orange glow. Farmers across centuries selected trees with thinner membranes, more juice, and reduced bitterness, turning an already successful hybrid into one of humanity's favorite fruit crops. Modern citrus genetics continues revealing just how influential this ancient hybridization event became."""),
("lemon","""Few ingredients have shaped world cuisine as profoundly as the lemon. Yet this fruit owes its existence to chance hybridization rather than a long independent history. The origin of lemons lies in South Asia, where citron and bitter orange naturally crossed. Instead of becoming sweeter over time, lemons retained remarkably high concentrations of citric acid, creating their unmistakable sharp taste. During ripening, yellow pigments gradually replace green chlorophyll, advertising maturity even before the fruit is picked. Sailors later discovered another valuable property: lemons dramatically reduced scurvy during long voyages. That discovery transformed the fruit from a regional crop into a global necessity, demonstrating how evolutionary chemistry unexpectedly influenced human history."""),
("lime","""Unlike many orchard crops that adapted to temperate climates, the lime remained loyal to the tropics. This fruit traces its origin to Southeast Asia, where warmth and humidity encouraged several closely related citrus species to evolve side by side. Many limes are harvested while still green, leading people to assume that green is their mature color, although fully ripe fruit often becomes yellow. Their intense acidity evolved partly as a defense before seeds matured, while aromatic oils attracted animals once ripening began. Human cultivation later emphasized abundant juice and fragrant peel, ensuring that limes became indispensable in cuisines from Mexico to Thailand."""),
("grapefruit","""Compared with apples or peaches, grapefruit is astonishingly young. The fruit first appeared in eighteenth-century Barbados after a sweet orange and a pomelo happened to hybridize naturally. That unexpected origin created a flavor unlike either parent—sweet, bitter, aromatic, and refreshing all at once. Ruby varieties later evolved through increased production of lycopene and carotenoid pigments, giving the flesh its striking color. Plant breeders worked to soften bitterness without erasing the fruit's distinctive personality. Grapefruit reminds botanists that evolution did not stop when agriculture began; entirely new cultivated fruits can still emerge through natural genetic combinations."""),
("mandarin","""Among citrus fruits, the mandarin occupies a place of unusual importance. Rather than being another hybrid, this fruit represents one of the original ancestral citrus lineages whose origin lies in southern China. Modern DNA studies reveal that countless oranges, tangelos, and other citrus hybrids inherited sweetness from mandarins. Easy-peeling skin and naturally high sugar content made them favorites wherever they were introduced. Their vivid orange appearance develops as carotenoid pigments accumulate during ripening, while relatively low acidity preserves an exceptionally gentle flavor. In many ways, understanding mandarin evolution is the key to understanding the entire citrus family."""),
("mango","""Picture an ancient tropical forest filled with elephants instead of orchards. That landscape shaped the early evolution of the mango. The enormous seed inside this fruit makes sense only when viewed alongside giant animals capable of carrying it across long distances. The origin of mango stretches across northeastern India, Bangladesh, and Myanmar, where forests nurtured extraordinary diversity. Wild mangoes were fibrous, resinous, and far less sweet than modern cultivars. Farmers patiently selected trees producing smoother flesh, richer aroma, and increasing sugar content until hundreds of distinct varieties emerged. During ripening, chlorophyll fades while yellow, orange, and crimson pigments dominate the skin. Today the mango remains one of the world's most celebrated tropical fruits because virtually every region has developed cultivars reflecting local tastes."""),
("cashew_apple","""The cashew apple begins with a botanical surprise. Although it looks like the obvious fruit, the colorful swollen structure is actually an enlarged stem, while the kidney-shaped shell attached beneath contains the true botanical fruit and seed. Its origin lies in northeastern Brazil, where evolution favored bright colors and abundant juice to attract animals capable of dispersing the attached seed. As ripening progresses, yellow, orange, or deep red pigments replace green, signaling maximum sweetness despite lingering astringency from natural tannins. Portuguese explorers later carried the cashew tree throughout the tropics, where people embraced both the edible nut and the juicy false fruit. Few plants challenge our everyday understanding of what a fruit really is quite as dramatically as the cashew.""")
]
for title, content in fruits:
    store.put(
        ("notes",),
        title,
        {"content": content}
    )

In [11]:
NAMESPACE = ("notes",)
d=store.search(NAMESPACE)
print(d)



[Item(namespace=['notes'], key='apple', value={'content': 'Few cultivated plants have traveled a journey as remarkable as the apple. The story of this fruit begins in the rugged Tian Shan Mountains of Central Asia, where forests of wild apples still grow today. The origin of nearly every modern apple can be traced back to these forests, where bears, horses, and other animals spread seeds long before humans cultivated orchards. As merchants carried apples along the Silk Road, trees naturally crossed with local relatives, creating extraordinary genetic diversity. Human preference then became a powerful evolutionary force. Farmers repeatedly chose trees that produced larger, sweeter, and crisper fruit, gradually transforming small tart apples into the familiar dessert varieties found today. Their brilliant red skin became increasingly common because anthocyanin pigments appealed to both wildlife and people, while green and yellow cultivars preserved older pigment combinations. Modern appl

In [47]:
# Iterating through all objects in a namespace
namespace_prefix = ("notes",)
offset=0
while True:
    page = store.search(namespace_prefix,offset=offset)
    
    if not page:
        break
        
    for item in page:
        print(f"Key: {item.key}, Value: {item.value}")
    offset += len(page)
        

Key: apple, Value: {'content': 'Apple is a sweet fruit'}


In [12]:
agent = create_agent(
    model=model,
    system_prompt="You are a NoteBot. each tool you have is designed for processing a single note you need to make multiple tool calls if you want to process more than one note.",
    tools=NOTEBOT_TOOLS,
    checkpointer=checkpointer,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                'save_note':True,
                'delete_note':{
                    "allowed_decisions":['approve','reject']
                }
            },
            description_prefix="Tool execution pending approval"
        ),
        ToolRetryMiddleware(
            max_retries=9,
            backoff_factor=1.1
        ),
        log_tool,
        log_and_trim_before_model,
    ],
    store=store
)

In [21]:
config = {"configurable": {"thread_id": "t1"}}

In [14]:
class MyCustomTransformer(StreamTransformer):
    required_stream_modes = ("custom",)

    def __init__(self, scope: tuple[str, ...] = ()) -> None:
        super().__init__(scope)
        # 1. Define the channel
        self.log = StreamChannel()

    def init(self) -> dict:
        # 2. Key "my_custom" will be the name used in interleave()
        return {"my_custom": self.log}

    def process(self, event) -> bool:
        if event["method"] == "custom":
            self.log.push(event["params"]["data"])
        return True

In [109]:
msg=input("Enter: ")
while(msg):
    res = agent.stream_events({"messages":[HumanMessage(msg)]},version='v3',config=config)
    for message in res.messages:
        for chunk in message.text:
            print(chunk,end="",flush=True)
        print("\n")
    msg=input("\nEnter: ")

In [20]:
msg = input("Enter: ")

while msg:
    stream = agent.stream_events(
        {"messages": [HumanMessage(msg)]},
        config=config,
        transformers=[MyCustomTransformer],
        version="v3",
    )
    for name, item in stream.interleave("my_custom", "messages"):
        if name == "my_custom":
            print(f"Tool update: {item}")
        elif name == "messages":
            for j in item.text:
                print(j,end="",flush=True)
    print()
    while stream.interrupted:
        interrupt = stream.interrupts[-1].value

        request = interrupt["action_requests"][-1]
        review = interrupt["review_configs"][-1]

        print("\nAction:")
        print(request["description"])

        print("\nAllowed decisions:")
        for d in review["allowed_decisions"]:
            print("-", d)

        choice = input("\nYour choice: ").strip()

        if choice == "approve":
            decision = {
                "type": "approve"
            }

        elif choice == "reject":
            feedback = input("Reason (optional): ")
            if feedback:
                feedback = f"Unsuccessful. The user has rejected the tool call with the following Feedback:{feedback}. Try Again"

            decision = {
                "type": "reject",
                "message": feedback
            }

        elif choice == "respond":
            reply = input("Response to tool: ")
            decision = {
                "type": "respond",
                "message": reply
            }

        elif choice == "edit":
            print(f"\nTool: {request['name']}")

            new_args = request["arguments"].copy()

            print("\nEdit arguments (press Enter to keep the current value):\n")

            for key, value in new_args.items():
                new_value = input(f"{key} [{value}]: ").strip()

                if new_value:
                    # Convert to original type where possible
                    try:
                        if isinstance(value, bool):
                            new_args[key] = new_value.lower() in (
                                "true",
                                "1",
                                "yes",
                                "y",
                            )
                        elif isinstance(value, int):
                            new_args[key] = int(new_value)
                        elif isinstance(value, float):
                            new_args[key] = float(new_value)
                        else:
                            new_args[key] = new_value
                    except ValueError:
                        print(f"Invalid value for {key}. Keeping original.")

                    decision = {
                        "type": "edit",
                        "edited_action": {
                            "name": request["name"],
                            "args": new_args,
                        },
                    }

        else:
            print("Invalid choice.")
            continue

        stream = agent.stream_events(
            Command(
                resume={
                    "decisions": [decision]
                }
            ),
            transformers=[MyCustomTransformer],
            config=config,
            version="v3",
        )

        for name, item in stream.interleave("my_custom", "messages"):
            if name == "my_custom":
                print(f"Tool update: {item}")
            elif name == "messages":
                for j in item.text:
                    print(j,end="",flush=True)

    msg = input("\nEnter: ")

About to call model with 3 messages
I’m not familiar with a tool called **summarize_tool**.  
If you’d like me to summarize the notes on “fruit”, I can use the `summarize_notes` tool instead. Let me know if that works for you!
About to call model with 5 messages
Executing tool: summarize_notes
Arguments: {'topic': 'fruit'}
Tool update: summarizing...
Tool update: Found 10 to summarize
Tool failed: model 'gemma4:e2bb-mlx' not found (status code: 404)
Executing tool: summarize_notes
Arguments: {'topic': 'fruit'}
Tool update: summarizing...
Tool update: Found 10 to summarize
Tool failed: model 'gemma4:e2bb-mlx' not found (status code: 404)
Executing tool: summarize_notes
Arguments: {'topic': 'fruit'}
Tool update: summarizing...
Tool update: Found 10 to summarize
Tool failed: model 'gemma4:e2bb-mlx' not found (status code: 404)
Executing tool: summarize_notes
Arguments: {'topic': 'fruit'}
Tool update: summarizing...
Tool update: Found 10 to summarize
Tool failed: model 'gemma4:e2bb-mlx' no

KeyboardInterrupt: 

In [23]:
pprint(agent.get_state(config=config).values)

{'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='b10cd924-3384-4222-a1ff-667a3e83f56a'),
              AIMessage(content=[{'type': 'text', 'text': 'I understand your frustration. However, as an AI NoteBot, I do not retain memory of our past conversations or personal details like your favorite color unless they are explicitly saved in the notes within this system.\n\nIf you would like me to acknowledge that yellow is your favorite color, please let me know!', 'index': 0}], additional_kwargs={}, response_metadata={'model': 'qwen3.5:4b-mlx', 'created_at': '2026-07-20T15:09:16.866129Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4850435917, 'load_duration': 45009250, 'prompt_eval_count': 1624, 'prompt_eval_duration': 617507042, 'eval_count': 61, 'eval_duration': 4165284833, 'logprobs': None, 'model_name': 'qwen3.5:4b-mlx', 'model_provider': 'ollama', 'output_version': 'v1'}, id='lc_run--019f8012-edcd-7e63-848c-67c1033fa32c', tool_calls=[]

In [ ]:
msg = input("Enter: ")

while msg:
    stream = agent.stream_events(
        {"messages": [HumanMessage(msg)]},
        config=config,
        version="v3",
    )

    for message in stream.messages:
        for token in message.text:
            print(token, end="", flush=True)
    print()
    
    while stream.interrupted:
        interrupt = stream.interrupts[-1].value

        request = interrupt["action_requests"][-1]
        review = interrupt["review_configs"][-1]

        print("\nAction:")
        print(request["description"])

        print("\nAllowed decisions:")
        for d in review["allowed_decisions"]:
            print("-", d)

        choice = input("\nYour choice: ").strip()

        if choice == "approve":
            decision = {
                "type": "approve"
            }

        elif choice == "reject":
            feedback = input("Reason (optional): ")
            if feedback:
                feedback = f"Unsuccessful. The user has rejected the tool call with the following Feedback:{feedback}. Try Again"

            decision = {
                "type": "reject",
                "message": feedback
            }

        elif choice == "respond":
            reply = input("Response to tool: ")
            decision = {
                "type": "respond",
                "message": reply
            }

        elif choice == "edit":
            print(f"\nTool: {request['name']}")

            new_args = request["arguments"].copy()

            print("\nEdit arguments (press Enter to keep the current value):\n")

            for key, value in new_args.items():
                new_value = input(f"{key} [{value}]: ").strip()

                if new_value:
                    # Convert to original type where possible
                    try:
                        if isinstance(value, bool):
                            new_args[key] = new_value.lower() in (
                                "true",
                                "1",
                                "yes",
                                "y",
                            )
                        elif isinstance(value, int):
                            new_args[key] = int(new_value)
                        elif isinstance(value, float):
                            new_args[key] = float(new_value)
                        else:
                            new_args[key] = new_value
                    except ValueError:
                        print(f"Invalid value for {key}. Keeping original.")

                    decision = {
                        "type": "edit",
                        "edited_action": {
                            "name": request["name"],
                            "args": new_args,
                        },
                    }

        else:
            print("Invalid choice.")
            continue

        stream = agent.stream_events(
            Command(
                resume={
                    "decisions": [decision]
                }
            ),
            config=config,
            version="v3",
        )

        for message in stream.messages:
            for token in message.text:
                print(token, end="", flush=True)
        print()

    msg = input("\nEnter: ")

In [17]:
pprint(_NOTE_STORE)

{}


In [42]:
pprint(agent.get_state(config=config).values)

{'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='77051a14-4a30-4ae8-bc4e-428880b9bc31'),
              AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you with your notes today? I can save, delete, search, or summarize notes for you.', 'index': 0}], additional_kwargs={}, response_metadata={'model': 'gemma4:e2b-mlx', 'created_at': '2026-07-19T06:10:50.262896Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8568990875, 'load_duration': 5500079500, 'prompt_eval_count': 345, 'prompt_eval_duration': 1947076708, 'eval_count': 26, 'eval_duration': 1091903625, 'logprobs': None, 'model_name': 'gemma4:e2b-mlx', 'model_provider': 'ollama', 'output_version': 'v1'}, id='lc_run--019f78ff-8d9c-7051-8354-b11ca0a85ffb', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 345, 'output_tokens': 26, 'total_tokens': 371}),
              HumanMessage(content='find a noted named apple', additional_kwargs={}, response_metada

In [30]:
msg=input("Enter: ")
while(msg):
    res = agent.stream_events({"messages":[HumanMessage(msg)]},version='v3',config=config)
    for message in res.messages:
        for chunk in message.text:
            print(chunk,end="",flush=True)
        print("\n")
        for chunk in message.tool_calls:
            print(chunk,end="",flush=True)
    if res.interrupted:
        print(res.interrupts[-1].value['action_requests'][-1]['description'])
        print("\n")
    msg=input("\nEnter: ")




{'type': 'tool_call_chunk', 'id': '1b94b66c-a709-46c8-b00d-d3f2a20f3eb7', 'name': 'search_notes', 'args': '{"query": "orange"}'}The note titled "orange" contains the following content:

"orange is both color and fruit"



In [11]:
pprint(agent.get_state(config=config).values)

{}


In [19]:
pprint(res.interrupts)

[Interrupt(value={'action_requests': [{'args': {'title': 'orange'},
                                       'description': 'Tool execution requires '
                                                      'approval\n'
                                                      '\n'
                                                      'Tool: delete_note\n'
                                                      "Args: {'title': "
                                                      "'orange'}",
                                       'name': 'delete_note'}],
                  'review_configs': [{'action_name': 'delete_note',
                                      'allowed_decisions': ['approve',
                                                            'edit',
                                                            'reject',
                                                            'respond']}]},
           id='51c8eba45f896313440fdac4e479c825')]


In [ ]:
print(res.interrupts[-1].value['action_requests'][-1]['description'])

Tool execution requires approval

Tool: delete_note
Args: {'title': 'orange'}


In [56]:
response = agent.stream_events(
    Command(
        resume={"decisions":[{"type":"approve"}]}
    ),
    config=config,
    version="v3"
)

for i in response.messages:
    for j in i.text:
        print(j,end="",flush=True)

The note titled "orange" has been successfully deleted. Is there anything else you'd like me to help you with?

In [59]:
pprint(agent.get_state(config=config).values)

{'messages': [HumanMessage(content='can you search for notes that have pisces in them?', additional_kwargs={}, response_metadata={}, id='d798e3c6-5d7b-4182-b86c-cb399a02c96d'),
              AIMessage(content=[{'type': 'tool_call', 'id': '1ef6865e-2656-4e66-a3fb-64cf6c679eaa', 'name': 'search_notes', 'args': {'query': 'pisces'}}], additional_kwargs={}, response_metadata={'model': 'qwen3.5:4b-mlx', 'created_at': '2026-07-11T06:36:55.769697Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5014120291, 'load_duration': 1947274041, 'prompt_eval_count': 579, 'prompt_eval_duration': 2095518458, 'eval_count': 25, 'eval_duration': 924863125, 'logprobs': None, 'model_name': 'qwen3.5:4b-mlx', 'model_provider': 'ollama', 'output_version': 'v1'}, id='lc_run--019f4fe4-9ec1-7003-a2f0-24404341ccb3', tool_calls=[{'name': 'search_notes', 'args': {'query': 'pisces'}, 'id': '1ef6865e-2656-4e66-a3fb-64cf6c679eaa', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 579, '

In [1]:
pprint(response.interrupted)

Pretty printing has been turned OFF


In [113]:
msg=input("Enter: ")
while(msg):
    res = agent.stream_events({"messages":[HumanMessage(msg)]},version='v3',config=config)
    for event in res:
        print("\n")
        print(event)
        print("\n")
    msg=input("\nEnter: ")



{'type': 'event', 'method': 'values', 'params': {'namespace': [], 'timestamp': 1783942697028, 'data': {'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='1d6fd6b8-ba8c-4c1e-baf5-d26a334ff77d')]}, 'interrupts': ()}, 'seq': 1}


About to call model with 2 messages


{'type': 'event', 'method': 'messages', 'params': {'namespace': [], 'timestamp': 1783942704776, 'data': ({'event': 'message-start', 'role': 'ai', 'id': 'lc_run--019f5b45-5047-7403-ad8f-6b83b7118674'}, {'ls_integration': 'langchain_chat_model', 'thread_id': 'test0', 'langgraph_step': 2, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:87d51003-3e87-d972-bd23-ebf90ec73ee4', 'checkpoint_ns': 'model:87d51003-3e87-d972-bd23-ebf90ec73ee4', 'ls_provider': 'ollama', 'ls_model_name': 'qwen3.5:4b-mlx', 'ls_model_type': 'chat', 'ls_temperature': 0.0, 'lc_versions': {'langchain-core': '1.4.8', 'lang